# 04 — Does the Vol Term Structure Predict Realised Volatility?

Having built the implied vol surface, we turn to an empirical question:

**Does the spread between long-dated and short-dated ATM implied vol predict**
**subsequent realised volatility?**

The intuition: if the market prices 3M vol significantly above 1M vol,
it may be anticipating rising volatility in the near future.
This is not a novel idea — the implied-realised vol relationship has been studied
extensively since Latane & Rendleman (1976) — but the *term structure* dimension
adds a forward-looking angle beyond the VIX-level signal.

## Research Design

- **Signal**: ATM implied vol spread = IV(90d) - IV(30d)
- **Target**: realised vol over the next 21 trading days (~1 month)
- **Period**: 2 years of daily data (SPY)
- **Method**: OLS regression with Newey-West standard errors (overlapping observations)

> **Upfront caveat**: overlapping 21-day windows induce serial correlation in the dependent variable.
> Standard OLS standard errors are biased; Newey-West correction is applied but imperfect.
> The regression is purely exploratory; no trading strategy is implied.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
from scipy import stats
from statsmodels.stats.sandwich_covariance import cov_hac
import statsmodels.api as sm

from src.utils import compute_realised_vol, compute_forward_realised_vol

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

## 1. Build the Historical Dataset

We use the VIX term structure as a proxy for the implied vol spread,
since computing the full surface historically would require expensive options data.

Specifically:
- **VIX** (^VIX): 30-day ATM implied vol for SPX
- **VIX3M** (^VIX3M): 90-day ATM implied vol for SPX
- **Spread**: VIX3M - VIX (positive = upward sloping, negative = inverted/stressed)

This is the closest freely available proxy for the 1M-3M spread on our surface.

In [ ]:
period = "3y"

# Helper: strip timezone & time-of-day so all indexes align by calendar date
def _normalize_index(s: pd.Series) -> pd.Series:
    s = s.copy()
    if s.index.tz is not None:
        s.index = s.index.tz_localize(None)
    s.index = s.index.normalize()  # strip time component, keep date only
    return s

# Download VIX and VIX3M
vix    = yf.Ticker("^VIX").history(period=period)["Close"].rename("vix")
vix3m  = yf.Ticker("^VIX3M").history(period=period)["Close"].rename("vix3m")

# Forward realised vol (21 trading days)
fwd_rv = compute_forward_realised_vol(ticker="SPY", horizon=21, period=period)

# Normalize all indexes to date-only, timezone-naive
vix    = _normalize_index(vix)
vix3m  = _normalize_index(vix3m)
fwd_rv = _normalize_index(fwd_rv)

# Align everything
df = pd.concat([vix, vix3m, fwd_rv], axis=1).dropna()
df["spread"] = (df["vix3m"] - df["vix"]) / 100.0  # in decimal
df["fwd_rv"] = df["fwd_realised_vol_21d"]           # already annualised decimal

print(f"Dataset: {len(df)} observations ({df.index[0].date()} to {df.index[-1].date()})")
print("\nDescriptive statistics:")
print(df[["vix", "vix3m", "spread", "fwd_rv"]].describe().round(4))

## 2. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# VIX levels
axes[0].plot(df.index, df["vix"], label="VIX (1M)", color="#2196F3", lw=1.5)
axes[0].plot(df.index, df["vix3m"], label="VIX3M (3M)", color="#FF9800", lw=1.5)
axes[0].set_ylabel("Implied Vol (%)")
axes[0].set_title("VIX vs VIX3M")
axes[0].legend()

# Spread
axes[1].bar(df.index, df["spread"] * 100, color=np.where(df["spread"] > 0, "#4CAF50", "#F44336"), alpha=0.7)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_ylabel("Spread (%) = VIX3M - VIX")
axes[1].set_title("Term Structure Spread (positive = upward sloping)")

# Forward RV
axes[2].plot(df.index, df["fwd_rv"] * 100, color="#9C27B0", lw=1.5)
axes[2].set_ylabel("Fwd Realised Vol (%)")
axes[2].set_title("SPY Realised Vol (next 21 trading days)")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig("../figures/04_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. The Spread-vs-Realised Scatter

A key diagnostic: does a wider spread (upward sloping term structure) correlate
with higher future realised vol?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(df["spread"] * 100, df["fwd_rv"] * 100,
           alpha=0.35, s=12, color="#2196F3", edgecolors="none")

# OLS line
x = df["spread"].values
y = df["fwd_rv"].values
slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line * 100, (intercept + slope * x_line) * 100,
        color="#F44336", lw=2, label=f"OLS: slope={slope:.2f}, R={r_val:.2f}")

ax.axvline(0, color="grey", ls=":", alpha=0.5)
ax.set_xlabel("VIX3M - VIX Spread (%)")
ax.set_ylabel("Fwd 21d Realised Vol (%)")
ax.set_title("Vol Spread vs Forward Realised Vol")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/04_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Pearson R: {r_val:.3f}, p-value: {p_val:.4f}, R-squared: {r_val**2:.3f}")

## 4. OLS Regression with Newey-West SE

We run a proper regression with Newey-West standard errors
to correct for serial correlation from overlapping 21-day windows.
The lags parameter is set to 21 (matching the horizon).

In [ ]:
X = sm.add_constant(df["spread"])
y = df["fwd_rv"]

ols = sm.OLS(y, X).fit()
nw_se = cov_hac(ols, nlags=21)
ols_nw = ols.get_robustcov_results(cov_type="HAC", maxlags=21)

print(ols_nw.summary())

## 5. Interpretation

The regression results should be read critically:

- **Sign of slope**: does the spread positively or negatively predict future RV?
  An inverted term structure (VIX > VIX3M) during stress periods often *precedes* high realised vol.
- **R-squared**: likely modest (0.05–0.20). Vol forecasting is genuinely hard.
  A low R-squared is expected and honest — not a sign the analysis failed.
- **Regime sensitivity**: run this split across high-VIX (>20) and low-VIX regimes.
  Relationships in vol markets are strongly regime-dependent.

In [ ]:
# Regime split: high-vol vs low-vol periods
vix_threshold = 20
high_vol = df[df["vix"] > vix_threshold]
low_vol  = df[df["vix"] <= vix_threshold]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, subset, label, color in [
    (axes[0], low_vol,  f"Low VIX (<={vix_threshold})",  "#4CAF50"),
    (axes[1], high_vol, f"High VIX (>{vix_threshold})",  "#F44336"),
]:
    x = subset["spread"].values
    y = subset["fwd_rv"].values
    slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
    ax.scatter(x * 100, y * 100, alpha=0.4, s=12, color=color, edgecolors="none")
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line * 100, (intercept + slope * x_line) * 100,
            color="black", lw=2, label=f"R={r_val:.2f}, slope={slope:.2f}")
    ax.set_title(f"{label} (n={len(subset)})")
    ax.set_xlabel("Spread (%)")
    ax.set_ylabel("Fwd Realised Vol (%)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/04_regime_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary & Conclusions

Key findings (results will vary by period):

- The term structure spread carries a modest signal for forward realised vol.
- The relationship is regime-dependent: high-VIX periods often show **reversed** dynamics
  (inverted term structure → elevated future RV).
- The in-sample R-squared is too low to support direct trading, but the signal
  retains value as a **risk-monitoring indicator** — particularly for detecting regime shifts.
- Extensions (not implemented here): out-of-sample testing, alternative predictors
  (skew, VIX-VVIX spread), controlling for macro variables.